<a href="https://colab.research.google.com/github/GokulM8/Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

For the Refresh / Content Opportunity Scoring lane, each row represents the search performance of a single webpage for a specific reporting period.

The analysis uses a mid-panel month (2026-03) to avoid using the final month as training data. This month provides historical information that can be used to build refresh recommendations without introducing future information into the model.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Feature

- ctr
- avg_position
- impressions
- content_age_days
- search_volume

## Label / Proxy

Refresh priority is approximated using future performance trend (declining vs. not declining).

## Context

- page_id
- month
- content_type

These fields help identify or group records but are not direct predictive features.

## Excluded

Future outcome variables and any label-derived columns are excluded because they would leak future information into the model.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
%pip install -q duckdb huggingface_hub

In [10]:
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
import duckdb
import pandas as pd

# Token comes from the Colab Secret, never printed — this repo is public.
HF_TOKEN = userdata.get("HF_TOKEN")

### Switching from row-by-row streaming to DuckDB

The `datasets` streaming loop was slow because it decodes every row in Python one at a time and re-scans from the start on every new pass. DuckDB reads the underlying parquet files directly with real column/row pruning, so filtering to March and aggregating happens inside DuckDB, not in a Python `for` loop. This is also the pattern the track points to on the assignment card — not BigQuery, which isn't used anywhere in this track.

In [11]:
# Find the parquet files that back this config, then pull them locally (gated repo, needs the token).
api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
config_files = sorted(f for f in all_files if "fact_content_daily_performance" in f and f.endswith(".parquet"))

print(f"Found {len(config_files)} parquet file(s) for this config:")
for f in config_files:
    print(" ", f)

Found 19 parquet file(s) for this config:
  fact_content_daily_performance/month=2025-01/data_0.parquet
  fact_content_daily_performance/month=2025-02/data_0.parquet
  fact_content_daily_performance/month=2025-03/data_0.parquet
  fact_content_daily_performance/month=2025-04/data_0.parquet
  fact_content_daily_performance/month=2025-05/data_0.parquet
  fact_content_daily_performance/month=2025-06/data_0.parquet
  fact_content_daily_performance/month=2025-07/data_0.parquet
  fact_content_daily_performance/month=2025-08/data_0.parquet
  fact_content_daily_performance/month=2025-09/data_0.parquet
  fact_content_daily_performance/month=2025-10/data_0.parquet
  fact_content_daily_performance/month=2025-11/data_0.parquet
  fact_content_daily_performance/month=2025-12/data_0.parquet
  fact_content_daily_performance/month=2026-01/data_0.parquet
  fact_content_daily_performance/month=2026-02/data_0.parquet
  fact_content_daily_performance/month=2026-03/data_0.parquet
  fact_content_daily_perform

In [13]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

remote_paths = [f"hf://datasets/FlyRank/internship-warehouse/{f}" for f in config_files]
paths_sql = "[" + ", ".join(f"'{p}'" for p in remote_paths) + "]"
con.sql(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet({paths_sql})")
con.sql("DESCRIBE fact").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

### Query 1 — grain check

*Claim: one row = one `(client_hash_id, content_hash_id, report_date)` combination — a group with `COUNT(*) > 1` breaks it.*

In [14]:
grain_check = con.sql("""
    SELECT COUNT(*) AS n_dupe_groups
    FROM (
        SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
        FROM fact
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()

print(grain_check)
print("Grain holds (0 duplicate groups expected):", grain_check['n_dupe_groups'][0] == 0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_dupe_groups
0              0
Grain holds (0 duplicate groups expected): True


### Query 2 — row count + date span

*Exact, over the full March slice — DuckDB scans it directly, no Python loop.*

In [15]:
query2 = con.sql("""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM fact
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

query2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — availability check (`IS TRUE`)

*Claim: not every warehouse row has real GSC data behind it.*

In [16]:
query3 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM fact
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

query3["survival_rate"] = query3["gsc_available_rows"] / query3["total_rows"]
query3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,survival_rate
0,9841378,3611061.0,0.366926


### Five features — monthly, content-level feature frame

*Aggregated in SQL over the full March month — no sampling cap needed now that DuckDB is doing the scan.*

In [17]:
features = con.sql("""
    WITH march AS (
        SELECT *, CAST(strftime(report_date, '%d') AS INTEGER) AS day
        FROM fact
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    ),
    with_sessions AS (
        SELECT *,
            sessions_organic + sessions_direct + sessions_referral
            + sessions_social + sessions_paid + sessions_ai AS total_sessions_row
        FROM march
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS total_impressions,
        SUM(sessions_ai) * 1.0 / NULLIF(SUM(total_sessions_row), 0) AS ai_search_share,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
        SUM(CASE WHEN day <= 15 THEN gsc_clicks ELSE 0 END) AS front_half_clicks,
        SUM(CASE WHEN day > 15 THEN gsc_clicks ELSE 0 END) AS back_half_clicks
    FROM with_sessions
    GROUP BY client_hash_id, content_hash_id
""").df().fillna(0)

print("Feature frame shape:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 9)


,client_hash_id,content_hash_id,avg_ctr,avg_position,total_impressions,ai_search_share,engagement_rate,front_half_clicks,back_half_clicks
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,0.000000,4.074107,77.0,0.0,0.0,0.0,0.0
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,0.006645,4.428747,602.0,0.0,0.0,2.0,2.0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,0.001235,4.866123,810.0,0.0,0.0,1.0,0.0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,0.000000,8.978086,82.0,0.0,0.0,0.0,0.0
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,0.003229,1.854929,1858.0,0.0,0.0,1.0,5.0


**Available when? — one line per feature:**

- `avg_ctr` — knowable at month-close: built only from clicks/impressions already logged in March, nothing from later months.
- `avg_position` — same: pulled from GSC rows already landed for March.
- `total_impressions` — same, a purely historical count.
- `ai_search_share` — same, GA4 session-source split already logged for March.
- `engagement_rate` — same, GA4 engaged-session data already landed for March.

None of the five touch April or June — all five are knowable the moment March closes, which is exactly when a refresh call would get made.

### The trap

*Add ONE label-derived column on purpose, watch the quick score jump toward perfect, then delete it and keep the honest number.*

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

features["is_declining"] = (features["back_half_clicks"] < features["front_half_clicks"]).astype(int)

honest_feature_cols = ["avg_ctr", "avg_position", "total_impressions", "ai_search_share", "engagement_rate"]

X = features[honest_feature_cols].fillna(0)
y = features["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, model.predict(X_test))
print(f"Honest quick score (5 real features): {honest_score:.3f}")

Honest quick score (5 real features): 0.911


In [19]:
# Deliberately leak the label: back_half_clicks is literally what is_declining was computed from.
leaky_feature_cols = honest_feature_cols + ["back_half_clicks"]

X_leak = features[leaky_feature_cols].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y, test_size=0.3, random_state=42, stratify=y
)

model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_train_l, y_train_l)
leaky_score = accuracy_score(y_test_l, model_leak.predict(X_test_l))

print(f"Leaky quick score (+ back_half_clicks): {leaky_score:.3f}")
print(f"Jump: {leaky_score - honest_score:+.3f}")

Leaky quick score (+ back_half_clicks): 0.912
Jump: +0.001


In [20]:
# Delete the leak. Keep the honest number.
del leaky_feature_cols, X_leak, X_train_l, X_test_l, y_train_l, y_test_l, model_leak

print("Leak column removed.")
print(f"The honest quick score stands at: {honest_score:.3f}")
print("That's the real number — not the one juiced by seeing the answer.")

Leak column removed.
The honest quick score stands at: 0.911
That's the real number — not the one juiced by seeing the answer.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

- **In-month split, not a true forward label.** `is_declining` compares the first half of March to the second half of the *same* month. It's a cheap way to demonstrate the leakage mechanic, not a real past→future label built from an actual following month — a production label would need April (or later) data, still respecting the sealed-June rule.
- **GSC/GA4 availability is uneven.** Not every client has both `client_has_gsc` and `client_has_ga4` true, and `gsc_data_available` / `ga4_data_available` can be false even for nominally onboarded clients (Query 3 shows the actual survival rate) — so this slice systematically favors clients with more complete instrumentation.
- **Single fact table, no dimension join.** Everything here comes from `fact_content_daily_performance` alone — nothing is joined against client vertical, plan tier, or contract size, so none of those differences are visible in the features or the label.

## Self-check

Before you submit, confirm each line honestly:

- [-] Every section above is filled — markdown thinking AND the code that backs it
- [-] The notebook runs top to bottom with no errors (Runtime → Run all)
- [-] No client names, URLs, or private queries anywhere
- [-] My claims use careful words: observed, measured, directional, decision-support
- [-] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.